In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'minisgl').exists():
    ROOT = ROOT.parent
if not (ROOT / 'minisgl').exists():
    raise FileNotFoundError('请从 mini-sglang 目录或其子目录启动 Jupyter')
sys.path.insert(0, str(ROOT))

from examples.shm_pinned_experiment import (
    benchmark_backend, benchmark_copy, to_rows, verify_protocol, verify_ring,
)


In [2]:
# 方案 9.1：协议、TransferKey/Meta 版本校验、槽位回绕和边界检查
print(verify_protocol())
print(verify_ring(payload_size=4096, slots=3))


{'transfer_key_round_trip': True, 'transfer_meta_round_trip': True, 'version_rejected': True}
{'wraparound': True, 'bounds_rejected': True, 'slots': 3}


In [3]:
# 方案 8/9：host shared-memory copy 微基准。单位 GiB/s 是十进制近似。
copy_rows = benchmark_copy(warmup=3, repeat=20, pinned=False)
copy_rows


[{'bytes': 1048576,
  'pinned': False,
  'shm_p50_ms': 0.29004737734794617,
  'plain_p50_ms': 0.2911500632762909,
  'shm_gib_per_s': 3.6151886963697963,
  'plain_gib_per_s': 3.6014967271531697},
 {'bytes': 4194304,
  'pinned': False,
  'shm_p50_ms': 0.7423069328069687,
  'plain_p50_ms': 0.7309690117835999,
  'shm_gib_per_s': 5.650363501442195,
  'plain_gib_per_s': 5.738005212786921},
 {'bytes': 16777216,
  'pinned': False,
  'shm_p50_ms': 2.081768587231636,
  'plain_p50_ms': 2.039549872279167,
  'shm_gib_per_s': 8.059116706295663,
  'plain_gib_per_s': 8.225940550917594}]

In [4]:
# 仅 CUDA 环境执行：测试 cudaHostRegister 后的 host ring。
import torch
if torch.cuda.is_available():
    pinned_copy_rows = benchmark_copy(warmup=3, repeat=20, pinned=True)
    pinned_copy_rows
else:
    print('跳过：当前环境没有 CUDA')


In [5]:
MODEL_PATH = '/root/autodl-tmp/LLM-sglang/Qwen/Qwen3-8B-FP8'
DTYPE = 'auto' 
DEVICES = (0, 1)
INPUT_LEN = 1024
OUTPUT_LEN =1024
BATCH_SIZE = 4
WARMUP = 5
REPEAT = 20
PAGE_SIZE = 16
CHUNK_TOKENS = 16
SLOT_COUNT = 4

assert Path(MODEL_PATH).exists(), '请先设置 MODEL_PATH'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, '端到端实验需要两张 CUDA GPU'


In [ ]:
results = []
for backend in ('none', 'shm', 'shm_pinned'):
    print(f'Running {backend} ...')
    results.append(benchmark_backend(
        MODEL_PATH, backend, input_len=INPUT_LEN, output_len=OUTPUT_LEN,
        batch_size=BATCH_SIZE, warmup=WARMUP, repeat=REPEAT,
        page_size=PAGE_SIZE, chunk_tokens=CHUNK_TOKENS,
        slot_count=SLOT_COUNT, devices=DEVICES, dtype=DTYPE,
    ))

rows = to_rows(results)
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    rows


In [8]:
# 相对 none 基线的变化：
# 正值表示提升，负值表示下降

if not rows:
    raise ValueError("rows 为空，没有可展示的实验结果")

baseline = next(
    (row for row in rows if row.get("backend") == "none"),
    None,
)

if baseline is None:
    raise ValueError("rows 中没有 backend='none' 的基线结果")

def percentage_change(value, baseline_value, higher_is_better=True):
    if baseline_value == 0:
        return 0.0

    if higher_is_better:
        return 100.0 * (value - baseline_value) / baseline_value
    else:
        return 100.0 * (baseline_value - value) / baseline_value


print("=" * 90)
print("实验结果：相对 none 基线的性能变化")
print("=" * 90)
print(f"基线 backend: none")
print(f"基线 TTFT P90: {baseline['ttft_p90_ms']:.2f} ms")
print(f"基线 TPOT P90: {baseline['tpot_p90_ms']:.2f} ms")
print(f"基线吞吐:      {baseline['output_tok_per_s']:.2f} tok/s")
print("-" * 90)

for row in rows:
    backend = row["backend"]

    # 延迟越低越好，因此基线减当前值
    ttft_delta_pct = percentage_change(
        row["ttft_p90_ms"],
        baseline["ttft_p90_ms"],
        higher_is_better=False,
    )

    tpot_delta_pct = percentage_change(
        row["tpot_p90_ms"],
        baseline["tpot_p90_ms"],
        higher_is_better=False,
    )

    # 吞吐越高越好，因此当前值减基线
    throughput_delta_pct = percentage_change(
        row["output_tok_per_s"],
        baseline["output_tok_per_s"],
        higher_is_better=True,
    )

    print(f"backend: {backend}")
    print(f"  success_rate:          {row['success_rate']:.2%}")
    print(f"  TTFT P90:              {row['ttft_p90_ms']:.2f} ms")
    print(f"  TTFT P90 变化:         {ttft_delta_pct:+.2f}%")
    print(f"  TPOT P90:              {row['tpot_p90_ms']:.2f} ms")
    print(f"  TPOT P90 变化:         {tpot_delta_pct:+.2f}%")
    print(f"  output throughput:     {row['output_tok_per_s']:.2f} tok/s")
    print(f"  throughput 变化:       {throughput_delta_pct:+.2f}%")
    print(f"  transfer_failures:     {row['transfer_failures']}")
    print("-" * 90)


实验结果：相对 none 基线的性能变化
基线 backend: none
基线 TTFT P90: 1215.92 ms
基线 TPOT P90: 60.28 ms
基线吞吐:      16.53 tok/s
------------------------------------------------------------------------------------------
backend: none
  success_rate:          100.00%
  TTFT P90:              1215.92 ms
  TTFT P90 变化:         +0.00%
  TPOT P90:              60.28 ms
  TPOT P90 变化:         +0.00%
  output throughput:     16.53 tok/s
  throughput 变化:       +0.00%
  transfer_failures:     0
------------------------------------------------------------------------------------------
backend: shm
  success_rate:          100.00%
  TTFT P90:              1783.10 ms
  TTFT P90 变化:         -46.65%
  TPOT P90:              69.25 ms
  TPOT P90 变化:         -14.89%
  output throughput:     15.19 tok/s
  throughput 变化:       -8.08%
  transfer_failures:     0
------------------------------------------------------------------------------------------
backend: shm_pinned
  success_rate:          100.00%
  TTFT P90:             

In [ ]:

results = []

result = benchmark_backend(
        MODEL_PATH, "queue", input_len=INPUT_LEN, output_len=OUTPUT_LEN,
        batch_size=BATCH_SIZE, warmup=WARMUP, repeat=REPEAT,
        page_size=PAGE_SIZE, chunk_tokens=CHUNK_TOKENS,
        slot_count=SLOT_COUNT, devices=DEVICES, dtype=DTYPE,
    )

results.append(result)
print(result)


In [7]:
print(result)

{'backend': 'queue', 'input_len': 1024, 'output_len': 1024, 'batch_size': 4, 'requests': 80, 'successful_requests': 80, 'success_rate': 1.0, 'ttft_p50_ms': 3865.0227524340153, 'ttft_p90_ms': 4146.423157304525, 'ttft_p99_ms': 4255.761343985796, 'tpot_p50_ms': 66.73781394406602, 'tpot_p90_ms': 86.11358348378744, 'tpot_p99_ms': 131.2747585626555, 'total_p50_ms': 36483.047439716756, 'total_p90_ms': 67919.90446019918, 'total_p99_ms': 68659.14967469871, 'output_tokens': 48414, 'output_tok_per_s': 14.142593714280618, 'transfer_failures': 0}


In [8]:
import nixl._api
print("NIXL import OK")
# NIXL + UCX 1P1D 性能测试：复用与其他 backend 完全相同的配置、warmup/repeat 和度量。
nixl_result = benchmark_backend(
    MODEL_PATH, "nixl", input_len=INPUT_LEN, output_len=OUTPUT_LEN,
    batch_size=BATCH_SIZE, warmup=WARMUP, repeat=REPEAT,
    page_size=PAGE_SIZE, chunk_tokens=CHUNK_TOKENS,
    slot_count=SLOT_COUNT, devices=DEVICES, dtype=DTYPE,
)

nixl_rows = to_rows([nixl_result])
try:
    import pandas as pd
    display(pd.DataFrame(nixl_rows))
except ImportError:
    nixl_rows

print(nixl_result)


NIXL import OK
[2026-10-01 18:28:24] [INFO] [minisgl] Creating model: Qwen3ForCausalLM (type=qwen3)
[2026-10-01 18:28:24] [INFO] [minisgl] Creating model: Qwen3ForCausalLM (type=qwen3)
[2026-10-01 18:29:26] [INFO] [minisgl] Loaded 399 weights (model_type=qwen3)
[2026-10-01 18:29:26] [INFO] [minisgl] Allocating KV cache: 524 pages
[2026-10-01 18:29:26] [INFO] [minisgl] Engine initialized on rank 0 (device=cuda)
2026-10-01 18:29:26 NIXL INFO    _api.py:285 Initialized NIXL agent: mini-sglang-prefill-0
2026-10-01 18:29:27 NIXL INFO    _api.py:407 Backend UCX was instantiated
[2026-10-01 18:29:27] [INFO] [minisgl] Loaded 399 weights (model_type=qwen3)
[2026-10-01 18:29:27] [INFO] [minisgl] Allocating KV cache: 524 pages
[2026-10-01 18:29:27] [INFO] [minisgl] Engine initialized on rank 0 (device=cuda)
2026-10-01 18:29:27 NIXL INFO    _api.py:285 Initialized NIXL agent: mini-sglang-decode-1
2026-10-01 18:29:29 NIXL INFO    _api.py:407 Backend UCX was instantiated
{'backend': 'nixl', 'input_l